# Mundari

In [ ]:
"""
Layer-wise Cosine Similarity Pipeline — XLSR-300M
===================================================
Loads Mundari audio directly from a folder of WAV files,
then compares how similarly each fine-tuned model encodes
that audio at every transformer layer vs. the Mundari model.

Layer zones (XLSR-300M, 25 hidden states total):
    Layer  0      → CNN feature extractor
    Layers  1–6   → Acoustic / spectral
    Layers  7–12  → Phonetic   ← key zone
    Layers 13–24  → Linguistic

Requirements:
    pip install transformers torch librosa
                scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← only edit this block
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    "models": {
        "Mundari": "model_path",
        "Santali": "model_path",
        "Bengali": "model_path",
        "Hindi": "model_path",
    },

    # Folder containing Mundari WAV files (searched recursively).
    # Can be any mix of .wav / .flac / .mp3 — librosa handles all.
    "audio_dir":   "data_test_path",   # ← set this path

    "num_samples":    500,       # max clips to use (randomly sampled if more exist)
    "num_layers":     25,        # CNN(1) + transformer(24) = 25
    "output_dir":     "outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,        # clips longer than this are truncated
}

# ─────────────────────────────────────────────────────────────────────────────
# Styling
# ─────────────────────────────────────────────────────────────────────────────

LANG_COLORS = {"Santali": "#E63946", "Bengali": "#2A9D8F", "Hindi": "#E9C46A"}
LANG_STYLES = {
    "Santali": {"linestyle": "-",  "linewidth": 2.5, "marker": "o", "markersize": 4},
    "Bengali": {"linestyle": "--", "linewidth": 2.5, "marker": "s", "markersize": 4},
    "Hindi":   {"linestyle": ":",  "linewidth": 2.5, "marker": "^", "markersize": 4},
}
_FC = ["#F4A261", "#A8DADC", "#457B9D", "#CDB4DB"]
_FS = [{"linestyle": "-.", "linewidth": 2.5, "marker": "D", "markersize": 4}]
BG  = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1–6)"),
    (7,  13, "#FF006E", "Phonetic (L7–12)"),
    (13, 25, "#FB5607", "Linguistic (L13–24)"),
]

def _color(lang, idx): return LANG_COLORS.get(lang, _FC[idx % len(_FC)])
def _style(lang, idx): return LANG_STYLES.get(lang, _FS[idx % len(_FS)])


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load WAV files
# ─────────────────────────────────────────────────────────────────────────────

def load_audio_from_dir(cfg):
    """
    Recursively finds all audio files under cfg['audio_dir'],
    randomly samples up to cfg['num_samples'], loads and peak-normalises them.
    Returns a list of 1-D float32 numpy arrays at 16 kHz.
    """
    audio_dir = cfg["audio_dir"]
    print(f"\n[1/4] Scanning audio files in: {audio_dir}")

    extensions = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(audio_dir, "**", ext), recursive=True))
    files = sorted(files)

    if not files:
        raise FileNotFoundError(
            f"No audio files found under '{audio_dir}'.\n"
            f"Searched for: {extensions}\n"
            f"Check CONFIG['audio_dir'] is correct."
        )

    print(f"  Found {len(files)} audio file(s)")

    # Random subsample
    random.seed(42)
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
        print(f"  Randomly sampled {cfg['num_samples']} files")

    max_len    = int(cfg["max_duration_s"] * cfg["sample_rate"])
    audio_list = []
    skipped    = 0

    for fp in tqdm(files, desc="  Loading"):
        try:
            arr, sr = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            arr = arr[:max_len]
            if len(arr) < 400:      # skip clips < ~25 ms
                skipped += 1
                continue
            peak = np.abs(arr).max()
            if peak > 0:
                arr = arr / peak    # peak-normalise
            audio_list.append(arr)
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"\n  [warn] {os.path.basename(fp)} skipped — {exc}")

    print(f"  Loaded {len(audio_list)} clips  ({skipped} skipped)")
    if not audio_list:
        raise RuntimeError("No clips loaded — check audio_dir and file formats.")
    return audio_list


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Layer-wise embeddings
# ─────────────────────────────────────────────────────────────────────────────

def extract_layerwise_embeddings(audio_list, model_path, cfg):
    """
    Passes every clip through the model and returns the grand-mean
    hidden state at every layer: np.ndarray of shape (num_layers, hidden_dim).
    """
    device = cfg["device"]
    print(f"    Model : {model_path}")

    fe    = Wav2Vec2FeatureExtractor.from_pretrained(model_path)
    model = Wav2Vec2Model.from_pretrained(model_path)
    model.eval().to(device)

    layer_accum = None
    count       = 0

    for audio in tqdm(audio_list, desc="    Extracting", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)

            # hidden_states: tuple of (1, T, D) tensors; length = num_layers
            clip = np.stack([
                hs.squeeze(0).mean(0).cpu().float().numpy()
                for hs in out.hidden_states
            ])  # (num_layers, D)

            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception as exc:
            print(f"\n    [warn] clip skipped — {exc}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if count == 0:
        raise RuntimeError(f"No embeddings extracted from {model_path}")
    return layer_accum / count   # (num_layers, D)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Per-layer cosine similarity
# ─────────────────────────────────────────────────────────────────────────────

def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Plots
# ─────────────────────────────────────────────────────────────────────────────

def plot_layerwise_similarity(similarity_dict, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers       = list(range(cfg["num_layers"]))
    layer_labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]

    # ── Figure 1: Line plot ──────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)

    for idx, (lang, sims) in enumerate(similarity_dict.items()):
        ax.plot(layers, sims, color=_color(lang,idx), label=lang, **_style(lang,idx))
        pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
        ax.annotate(f"  {pv:.3f}", xy=(pl, pv),
                    color=_color(lang,idx), fontsize=8, fontweight="bold")

    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Mundari", color="#CCCCCC", fontsize=12)
    ax.set_title(
        "Layer-wise Acoustic Similarity with Mundari\n"
        "(XLSR-300M — Mundari audio encoded by each fine-tuned model)",
        color="white", fontsize=14, fontweight="bold", pad=15)
    ax.set_xticks(layers)
    ax.set_xticklabels(layer_labels, rotation=45, color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")

    lh = [plt.Line2D([0],[0], color=_color(l,i), label=l, **_style(l,i))
          for i,l in enumerate(similarity_dict)]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _,_,c,lb in ZONES]
    ax.legend(handles=lh+zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")

    plt.tight_layout()
    out1 = f"{cfg['output_dir']}/layerwise_similarity.png"
    plt.savefig(out1, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out1}")

    # ── Figure 2: Heatmap ────────────────────────────────────────────────────
    df_h = pd.DataFrame(similarity_dict, index=layer_labels)
    fig, ax = plt.subplots(figsize=(max(6, len(similarity_dict)*3), 12))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    sns.heatmap(df_h, ax=ax, cmap="RdYlGn",
                vmin=round(df_h.values.min()-0.02, 2), vmax=1.0,
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.4, linecolor="#222222",
                cbar_kws={"shrink": 0.6, "label": "Cosine Similarity"})
    ax.set_title("Cosine Similarity Heatmap\n(layer × language vs. Mundari)",
                 color="white", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(colors="#CCCCCC", labelsize=8)
    ax.set_xlabel("Language Model", color="#CCCCCC", fontsize=10)
    ax.set_ylabel("Layer", color="#CCCCCC", fontsize=10)
    cb = ax.collections[0].colorbar
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    plt.tight_layout()
    out2 = f"{cfg['output_dir']}/similarity_heatmap.png"
    plt.savefig(out2, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out2}")

    # ── Figure 3: Zone bar chart ─────────────────────────────────────────────
    zs   = {"Acoustic\n(L1–6)": slice(1,7),
            "Phonetic\n(L7–12)": slice(7,13),
            "Linguistic\n(L13–24)": slice(13,25)}
    zn   = list(zs.keys()); langs = list(similarity_dict.keys())
    nl   = len(langs); tw = 0.7; bw = tw/nl
    offs = np.linspace(-tw/2+bw/2, tw/2-bw/2, nl)
    x    = np.arange(len(zn))

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for idx, lang in enumerate(langs):
        sims = similarity_dict[lang]
        vals = [float(np.mean(sims[s])) for s in zs.values()]
        bars = ax.bar(x+offs[idx], vals, bw, label=lang,
                      color=_color(lang,idx), alpha=0.85, edgecolor="#222222")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f"{val:.3f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels(zn, color="#CCCCCC", fontsize=11)
    ax.set_ylabel("Mean Cosine Similarity", color="#CCCCCC", fontsize=11)
    ax.set_title("Zone-averaged Similarity with Mundari\n"
                 "(Acoustic / Phonetic / Linguistic layers)",
                 color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, axis="y", alpha=0.15, color="#FFFFFF", linestyle="--")
    ax.set_ylim(0, 1.08)
    ax.legend(facecolor="#1A1A2E", edgecolor="#444444",
              labelcolor="white", fontsize=10)
    plt.tight_layout()
    out3 = f"{cfg['output_dir']}/zone_similarity_bars.png"
    plt.savefig(out3, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out3}")
    return out1, out2, out3


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Summary table
# ─────────────────────────────────────────────────────────────────────────────

def print_summary_table(similarity_dict, cfg):
    zs = {"CNN (L0)":            slice(0, 1),
          "Acoustic (L1–6)":     slice(1, 7),
          "Phonetic (L7–12)":    slice(7, 13),
          "Linguistic (L13–24)": slice(13, 25),
          "Overall mean":        slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        row["Peak layer"] = int(np.argmax(sims))
        row["Peak value"] = round(float(np.max(sims)), 4)
        rows.append(row)
    df  = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 75
    print(f"\n{sep}\n  LAYER-WISE COSINE SIMILARITY SUMMARY\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/similarity_summary.csv"
    df.to_csv(csv); print(f"\n  Summary CSV → {csv}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    cfg = CONFIG
    print(f"\nDevice  : {cfg['device']}")
    print(f"Models  : {list(cfg['models'].keys())}")

    audio_list = load_audio_from_dir(cfg)

    print("\n[2/4] Extracting embeddings …")
    print("  → Reference: Mundari")
    ref_emb = extract_layerwise_embeddings(audio_list, cfg["models"]["Mundari"], cfg)

    compare_langs   = [l for l in cfg["models"] if l != "Mundari"]
    similarity_dict = {}
    for lang in compare_langs:
        print(f"  → Comparison: {lang}")
        other_emb = extract_layerwise_embeddings(audio_list, cfg["models"][lang], cfg)
        sims = compute_layerwise_cosine(ref_emb, other_emb)
        similarity_dict[lang] = sims
        print(f"     Phonetic zone mean (L7–12) : {np.mean(sims[7:13]):.4f}")

    print("\n[3/4] Generating plots …")
    plot_layerwise_similarity(similarity_dict, cfg)

    print("\n[4/4] Summary")
    print_summary_table(similarity_dict, cfg)
    print(f"\n✓ Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

# Bhili

In [ ]:
"""
Layer-wise Cosine Similarity Pipeline — XLSR-300M
===================================================
Loads Bhili audio directly from a folder of WAV files,
then compares how similarly each fine-tuned model encodes
that audio at every transformer layer vs. the Bhili model.

Layer zones (XLSR-300M, 25 hidden states total):
    Layer  0      → CNN feature extractor
    Layers  1–6   → Acoustic / spectral
    Layers  7–12  → Phonetic   ← key zone
    Layers 13–24  → Linguistic

Requirements:
    pip install transformers torch librosa
                scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← only edit this block
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    "models": {
        "Bhili": "model_path",
        "Gujarati": "model_path",
        "Marathi": "model_path",
        "Hindi": "model_path",
    },

    # Folder containing Bhili WAV files (searched recursively).
    # Can be any mix of .wav / .flac / .mp3 — librosa handles all.
    "audio_dir":   "data_test_path",   # ← set this path

    "num_samples":    500,       # max clips to use (randomly sampled if more exist)
    "num_layers":     25,        # CNN(1) + transformer(24) = 25
    "output_dir":     "outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,        # clips longer than this are truncated
}

# ─────────────────────────────────────────────────────────────────────────────
# Styling
# ─────────────────────────────────────────────────────────────────────────────

LANG_COLORS = {"Gujarati": "#E63946", "Marathi": "#2A9D8F", "Hindi": "#E9C46A"}
LANG_STYLES = {
    "Gujarati": {"linestyle": "-",  "linewidth": 2.5, "marker": "o", "markersize": 4},
    "Marathi": {"linestyle": "--", "linewidth": 2.5, "marker": "s", "markersize": 4},
    "Hindi":   {"linestyle": ":",  "linewidth": 2.5, "marker": "^", "markersize": 4},
}
_FC = ["#F4A261", "#A8DADC", "#457B9D", "#CDB4DB"]
_FS = [{"linestyle": "-.", "linewidth": 2.5, "marker": "D", "markersize": 4}]
BG  = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1–6)"),
    (7,  13, "#FF006E", "Phonetic (L7–12)"),
    (13, 25, "#FB5607", "Linguistic (L13–24)"),
]

def _color(lang, idx): return LANG_COLORS.get(lang, _FC[idx % len(_FC)])
def _style(lang, idx): return LANG_STYLES.get(lang, _FS[idx % len(_FS)])


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load WAV files
# ─────────────────────────────────────────────────────────────────────────────

def load_audio_from_dir(cfg):
    """
    Recursively finds all audio files under cfg['audio_dir'],
    randomly samples up to cfg['num_samples'], loads and peak-normalises them.
    Returns a list of 1-D float32 numpy arrays at 16 kHz.
    """
    audio_dir = cfg["audio_dir"]
    print(f"\n[1/4] Scanning audio files in: {audio_dir}")

    extensions = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(audio_dir, "**", ext), recursive=True))
    files = sorted(files)

    if not files:
        raise FileNotFoundError(
            f"No audio files found under '{audio_dir}'.\n"
            f"Searched for: {extensions}\n"
            f"Check CONFIG['audio_dir'] is correct."
        )

    print(f"  Found {len(files)} audio file(s)")

    # Random subsample
    random.seed(42)
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
        print(f"  Randomly sampled {cfg['num_samples']} files")

    max_len    = int(cfg["max_duration_s"] * cfg["sample_rate"])
    audio_list = []
    skipped    = 0

    for fp in tqdm(files, desc="  Loading"):
        try:
            arr, sr = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            arr = arr[:max_len]
            if len(arr) < 400:      # skip clips < ~25 ms
                skipped += 1
                continue
            peak = np.abs(arr).max()
            if peak > 0:
                arr = arr / peak    # peak-normalise
            audio_list.append(arr)
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"\n  [warn] {os.path.basename(fp)} skipped — {exc}")

    print(f"  Loaded {len(audio_list)} clips  ({skipped} skipped)")
    if not audio_list:
        raise RuntimeError("No clips loaded — check audio_dir and file formats.")
    return audio_list


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Layer-wise embeddings
# ─────────────────────────────────────────────────────────────────────────────

def extract_layerwise_embeddings(audio_list, model_path, cfg):
    """
    Passes every clip through the model and returns the grand-mean
    hidden state at every layer: np.ndarray of shape (num_layers, hidden_dim).
    """
    device = cfg["device"]
    print(f"    Model : {model_path}")

    fe    = Wav2Vec2FeatureExtractor.from_pretrained(model_path)
    model = Wav2Vec2Model.from_pretrained(model_path)
    model.eval().to(device)

    layer_accum = None
    count       = 0

    for audio in tqdm(audio_list, desc="    Extracting", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)

            # hidden_states: tuple of (1, T, D) tensors; length = num_layers
            clip = np.stack([
                hs.squeeze(0).mean(0).cpu().float().numpy()
                for hs in out.hidden_states
            ])  # (num_layers, D)

            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception as exc:
            print(f"\n    [warn] clip skipped — {exc}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if count == 0:
        raise RuntimeError(f"No embeddings extracted from {model_path}")
    return layer_accum / count   # (num_layers, D)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Per-layer cosine similarity
# ─────────────────────────────────────────────────────────────────────────────

def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Plots
# ─────────────────────────────────────────────────────────────────────────────

def plot_layerwise_similarity(similarity_dict, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers       = list(range(cfg["num_layers"]))
    layer_labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]

    # ── Figure 1: Line plot ──────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)

    for idx, (lang, sims) in enumerate(similarity_dict.items()):
        ax.plot(layers, sims, color=_color(lang,idx), label=lang, **_style(lang,idx))
        pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
        ax.annotate(f"  {pv:.3f}", xy=(pl, pv),
                    color=_color(lang,idx), fontsize=8, fontweight="bold")

    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Bhili", color="#CCCCCC", fontsize=12)
    ax.set_title(
        "Layer-wise Acoustic Similarity with Bhili\n"
        "(XLSR-300M — Bhili audio encoded by each fine-tuned model)",
        color="white", fontsize=14, fontweight="bold", pad=15)
    ax.set_xticks(layers)
    ax.set_xticklabels(layer_labels, rotation=45, color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")

    lh = [plt.Line2D([0],[0], color=_color(l,i), label=l, **_style(l,i))
          for i,l in enumerate(similarity_dict)]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _,_,c,lb in ZONES]
    ax.legend(handles=lh+zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")

    plt.tight_layout()
    out1 = f"{cfg['output_dir']}/layerwise_similarity.png"
    plt.savefig(out1, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out1}")

    # ── Figure 2: Heatmap ────────────────────────────────────────────────────
    df_h = pd.DataFrame(similarity_dict, index=layer_labels)
    fig, ax = plt.subplots(figsize=(max(6, len(similarity_dict)*3), 12))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    sns.heatmap(df_h, ax=ax, cmap="RdYlGn",
                vmin=round(df_h.values.min()-0.02, 2), vmax=1.0,
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.4, linecolor="#222222",
                cbar_kws={"shrink": 0.6, "label": "Cosine Similarity"})
    ax.set_title("Cosine Similarity Heatmap\n(layer × language vs. Bhili)",
                 color="white", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(colors="#CCCCCC", labelsize=8)
    ax.set_xlabel("Language Model", color="#CCCCCC", fontsize=10)
    ax.set_ylabel("Layer", color="#CCCCCC", fontsize=10)
    cb = ax.collections[0].colorbar
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    plt.tight_layout()
    out2 = f"{cfg['output_dir']}/similarity_heatmap.png"
    plt.savefig(out2, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out2}")

    # ── Figure 3: Zone bar chart ─────────────────────────────────────────────
    zs   = {"Acoustic\n(L1–6)": slice(1,7),
            "Phonetic\n(L7–12)": slice(7,13),
            "Linguistic\n(L13–24)": slice(13,25)}
    zn   = list(zs.keys()); langs = list(similarity_dict.keys())
    nl   = len(langs); tw = 0.7; bw = tw/nl
    offs = np.linspace(-tw/2+bw/2, tw/2-bw/2, nl)
    x    = np.arange(len(zn))

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for idx, lang in enumerate(langs):
        sims = similarity_dict[lang]
        vals = [float(np.mean(sims[s])) for s in zs.values()]
        bars = ax.bar(x+offs[idx], vals, bw, label=lang,
                      color=_color(lang,idx), alpha=0.85, edgecolor="#222222")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f"{val:.3f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels(zn, color="#CCCCCC", fontsize=11)
    ax.set_ylabel("Mean Cosine Similarity", color="#CCCCCC", fontsize=11)
    ax.set_title("Zone-averaged Similarity with Bhili\n"
                 "(Acoustic / Phonetic / Linguistic layers)",
                 color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, axis="y", alpha=0.15, color="#FFFFFF", linestyle="--")
    ax.set_ylim(0, 1.08)
    ax.legend(facecolor="#1A1A2E", edgecolor="#444444",
              labelcolor="white", fontsize=10)
    plt.tight_layout()
    out3 = f"{cfg['output_dir']}/zone_similarity_bars.png"
    plt.savefig(out3, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved → {out3}")
    return out1, out2, out3


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Summary table
# ─────────────────────────────────────────────────────────────────────────────

def print_summary_table(similarity_dict, cfg):
    zs = {"CNN (L0)":            slice(0, 1),
          "Acoustic (L1–6)":     slice(1, 7),
          "Phonetic (L7–12)":    slice(7, 13),
          "Linguistic (L13–24)": slice(13, 25),
          "Overall mean":        slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        row["Peak layer"] = int(np.argmax(sims))
        row["Peak value"] = round(float(np.max(sims)), 4)
        rows.append(row)
    df  = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 75
    print(f"\n{sep}\n  LAYER-WISE COSINE SIMILARITY SUMMARY\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/similarity_summary.csv"
    df.to_csv(csv); print(f"\n  Summary CSV → {csv}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    cfg = CONFIG
    print(f"\nDevice  : {cfg['device']}")
    print(f"Models  : {list(cfg['models'].keys())}")

    audio_list = load_audio_from_dir(cfg)

    print("\n[2/4] Extracting embeddings …")
    print("  → Reference: Bhili")
    ref_emb = extract_layerwise_embeddings(audio_list, cfg["models"]["Bhili"], cfg)

    compare_langs   = [l for l in cfg["models"] if l != "Bhili"]
    similarity_dict = {}
    for lang in compare_langs:
        print(f"  → Comparison: {lang}")
        other_emb = extract_layerwise_embeddings(audio_list, cfg["models"][lang], cfg)
        sims = compute_layerwise_cosine(ref_emb, other_emb)
        similarity_dict[lang] = sims
        print(f"     Phonetic zone mean (L7–12) : {np.mean(sims[7:13]):.4f}")

    print("\n[3/4] Generating plots …")
    plot_layerwise_similarity(similarity_dict, cfg)

    print("\n[4/4] Summary")
    print_summary_table(similarity_dict, cfg)
    print(f"\n✓ Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()